In [ ]:
#!pip install pandas
#!pip install ipython-sql prettytable

import prettytable

prettytable.DEFAULT = 'DEFAULT'

In [ ]:
import pandas as pd
import sqlite3

# Creating Database
conn = sqlite3.connect("AmazonDB.db")
amazon_df = pd.read_csv("AmazonData_SQL_queries.csv")

amazon_df.to_sql("AMAZON_DATA", conn, if_exists="replace", index=False)

conn.close()

Loading SQL magic module

In [ ]:
!pip install ipython-sql
%load_ext sql


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Connect to SQLite Database

Establish a connection between SQL magic module and the database (AMAZON DB)

In [ ]:
%sql sqlite:///AmazonDB.db

In [ ]:
# Displaying the table

%sql select * from AMAZON_DATA limit 5;

 * sqlite:///AmazonDB.db
Done.


product_id,product_name,final_price,actual_price,discount_percentage,rating,rating_count,roots,Category,Sub_category,Sub-sub_category,Product_type
B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",399.0,1099.0,64,4.2,24269.0,Computers & Accessories,Accessories & Peripherals,Cables & Accessories,Cables,USBCables
B098NS6PVG,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)",199.0,349.0,43,4.0,43994.0,Computers & Accessories,Accessories & Peripherals,Cables & Accessories,Cables,USBCables
B096MSW6CT,"Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices",199.0,1899.0,90,3.9,7928.0,Computers & Accessories,Accessories & Peripherals,Cables & Accessories,Cables,USBCables
B08HDJ86NZ,"boAt Deuce USB 300 2 in 1 Type-C & Micro USB Stress Resistant, Tangle-Free, Sturdy Cable with 3A Fast Charging & 480mbps Data Transmission, 10000+ Bends Lifespan and Extended 1.5m Length(Martian Red)",329.0,699.0,53,4.2,94363.0,Computers & Accessories,Accessories & Peripherals,Cables & Accessories,Cables,USBCables
B08CF3B7N1,"Portronics Konnect L 1.2M Fast Charging 3A 8 Pin USB Cable with Charge & Sync Function for iPhone, iPad (Grey)",154.0,399.0,61,4.2,16905.0,Computers & Accessories,Accessories & Peripherals,Cables & Accessories,Cables,USBCables


Customer Engagement and Product Positioning Analysis

How can a company improve product positioning and pricing strategy using customer engagement through ratings?

There are three different layers

1. Department-level

2. Category-level

3. Product-level

1. Department-level analysis

The main focus will be on which department has the most engagement with customers and whether they are satisfied or not.

Key Attributes (Columns)
1. Departments (roots)
2. Ratings
3. Rating count
4. Discount price


In [ ]:
# Query 1: Customers engagement
%%sql
select roots, sum(rating_count) as total_engagement
from AMAZON_DATA
group by roots
order by total_engagement desc;

 * sqlite:///AmazonDB.db
Done.


roots,total_engagement
Electronics,15778848.0
Computers & Accessories,7744153.0
Home & Kitchen,2991069.0
OfficeProducts,149675.0
Health & PersonalCare,3663.0


Observation Query 1:

Electronics has the highest engagement with the customers. Following with the computers and Accessories.

In [ ]:
# Query 2: Highest average rating
%%sql
select roots, avg(rating) as avg_rating
from AMAZON_DATA
group by roots
order by avg_rating desc;

 * sqlite:///AmazonDB.db
Done.


roots,avg_rating
OfficeProducts,4.309677419354838
Computers & Accessories,4.154966887417214
Electronics,4.081749049429652
Home & Kitchen,4.040715883668903
Health & PersonalCare,4.0


Observation Query 2:

Office Products Department has the highest average ratings.

In [ ]:
# Query 3: Deparments performing above global average rating
%%sql
select avg(rating)  as global_avg_ratings
from AMAZON_DATA

 * sqlite:///AmazonDB.db
Done.


global_avg_ratings
4.0966694420039


In [ ]:
# Query 3: Deparments performing above global average rating
%%sql
select roots, avg(rating) as avg_rating
from AMAZON_DATA
group by roots
having avg_rating > (select avg(rating)  as global_avg_ratings
from AMAZON_DATA)
order by avg_rating desc;

 * sqlite:///AmazonDB.db
Done.


roots,avg_rating
OfficeProducts,4.309677419354838
Computers & Accessories,4.154966887417214


Observation Query 3:

Office Products and Computer & Accessories show ratings above the global average, indicating stronger customer satisfaction.

In [ ]:
# Query 4: Revenue contribution per department
# NOTE: In the dataset there is no such column given which can be get the exact revenue
# In current case the rating count is consider as quantity which are sold
# This is just for assumption not the actual revenue.
# Revenue after discount is being calculated
%%sql

select roots, sum(final_price * rating_count) as approx_revenue
from AMAZON_DATA
group by roots
order by approx_revenue desc;

 * sqlite:///AmazonDB.db
Done.


roots,approx_revenue
Electronics,59179753425.0
Computers & Accessories,6358032883.47
Home & Kitchen,6264336175.799999
OfficeProducts,45786730.0
Health & PersonalCare,3293037.0


In [ ]:
# Over() window function
%%sql
select roots, sum(final_price * rating_count) over() as approx_revenue
from AMAZON_DATA
order by approx_revenue desc
limit 1;

 * sqlite:///AmazonDB.db
Done.


roots,approx_revenue
Computers & Accessories,71851202251.27


Key Note:

A window function with over() performs calculations across rows without collapsing them into groups.

It keeps all original rows and adds the calculated value to each row.

While group by performs calculations based on groups and collapses rows into one row per group.

In [ ]:
# Query 4: Revenue contribution per department (Sub-queries)
%%sql
select roots, sum(final_price * rating_count) as approx_revenue,
round
(sum(final_price * rating_count) * 100 /
(select(sum(final_price * rating_count))
from AMAZON_DATA),2) as revenue_percentage
from AMAZON_DATA
group by roots
order by approx_revenue desc

 * sqlite:///AmazonDB.db
Done.


roots,approx_revenue,revenue_percentage
Electronics,59179753425.0,82.36
Computers & Accessories,6358032883.47,8.85
Home & Kitchen,6264336175.799999,8.72
OfficeProducts,45786730.0,0.06
Health & PersonalCare,3293037.0,0.0


In [ ]:
# Query 4: Revenue contribution per department (Over) (Difficult to understand by I have kept as option 2)
%%sql
select roots, sum(final_price * rating_count) as approx_revenue,
round
(sum(final_price * rating_count) * 100.0 /
sum(sum(final_price * rating_count)) over(),2) as revenue_percentage
from AMAZON_DATA
group by roots
order by approx_revenue desc;

 * sqlite:///AmazonDB.db
Done.


roots,approx_revenue,revenue_percentage
Electronics,59179753425.0,82.36
Computers & Accessories,6358032883.47,8.85
Home & Kitchen,6264336175.799999,8.72
OfficeProducts,45786730.0,0.06
Health & PersonalCare,3293037.0,0.0


Observation Query 4:

Electronics has the highest approximate revenue shares. With majority shares of 82%.

In [ ]:
#Query 5: Ranking departments by engagement
# Rank window function is used to rank the department based on the total engagements
%%sql
select roots, sum(rating_count) as total_engagement,
rank() over(order by sum(rating_count) desc) as Rank_by_engagement
from AMAZON_DATA
group by roots;

 * sqlite:///AmazonDB.db
Done.


roots,total_engagement,Rank_by_engagement
Electronics,15778848.0,1
Computers & Accessories,7744153.0,2
Home & Kitchen,2991069.0,3
OfficeProducts,149675.0,4
Health & PersonalCare,3663.0,5


Observation Query 5:

Electronics as has ranked 1 as the highest engagement by the customers (by calculating total rating counts)


2. Category-level analysis

To analyse the performance of the the categories Sub-devisions of the deparments.

Columns needed:

1. Category
2. Final price
3. Actual price
4. Discount percentage
5. Ratings
6. Rating count

In [ ]:
# Query 6: Average price and rating per category
%%sql
select Category, round(avg(final_price),2) as avg_final_price, round(avg(rating),2) as global_avg_ratings
from AMAZON_DATA
group by Category
order by avg_final_price desc
Limit 5;

 * sqlite:///AmazonDB.db
Done.


Category,avg_final_price,global_avg_ratings
Laptops,37247.0,4.0
Tablets,26999.0,4.6
"HomeTheater,TV & Video",10407.12,4.08
Monitors,8199.0,4.25
Mobiles & Accessories,7134.05,4.13


In [ ]:
# Query 7: Categories with high discount but low rating
%%sql
select Category, round(avg(discount_percentage),2) as avg_discount_percentage, round(avg(rating),2) as avg_rating
from AMAZON_DATA
group by Category
having avg_discount_percentage >(select round(avg(discount_percentage),2) from AMAZON_DATA)
and avg_rating < (select round(avg(rating),2) from AMAZON_DATA)
order by avg_discount_percentage desc;

 * sqlite:///AmazonDB.db
Done.


Category,avg_discount_percentage,avg_rating
WearableTechnology,69.82,4.03
"Headphones,Earbuds & Accessories",59.53,3.93
HomeMedicalSupplies & Equipment,53.0,4.0
HomeAudio,49.69,4.07
"HomeTheater,TV & Video",49.47,4.08


Observation Query 7:

The overall avergae rating it around 4.09 so the Wearable Technology has the highest discount percentage with the low average rating.

In [ ]:
# Query 8: Category Revenue Share %
%%sql
select roots, Category, sum(final_price * rating_count) as approx_revenue,
round
(sum(final_price * rating_count) * 100.0 /
sum(sum(final_price * rating_count)) over(Partition by roots),2) as revenue_percentage
from AMAZON_DATA
group by roots, Category
order by revenue_percentage desc;

 * sqlite:///AmazonDB.db
Done.


roots,Category,approx_revenue,revenue_percentage
Health & PersonalCare,HomeMedicalSupplies & Equipment,3293037.0,100.0
Home & Kitchen,Kitchen & HomeAppliances,4594223297.25,73.34
OfficeProducts,OfficePaperProducts,25813782.0,56.38
Electronics,Mobiles & Accessories,31471203446.0,53.18
OfficeProducts,OfficeElectronics,19972948.0,43.62
Computers & Accessories,Accessories & Peripherals,2380882173.4700003,37.45
Electronics,"HomeTheater,TV & Video",18242991811.0,30.83
Computers & Accessories,ExternalDevices & DataStorage,1899599928.0,29.88
Home & Kitchen,"Heating,Cooling & AirQuality",1582809859.11,25.27
Computers & Accessories,NetworkingDevices,1474685438.0,23.19


Observation Query 8:

Revenue in the Home & Kitchen department is highly concentrated, with Kitchen & HomeAppliances alone contributing over 73% of total department revenue, indicating a dominant product segment.

The Electronics department shows moderate revenue concentration, with Mobiles & Accessories generating over 53% of revenue, followed by HomeTheater, TV & Video at 30.83%, suggesting strong dependence on two primary segments.

Key Note:

Revenue distribution across departments follows a highly skewed structure. In most departments, a small number of categories account for the majority of revenue, reflecting a Pareto-like distribution. This suggests potential strategic focus areas for investment and marketing optimization.

In [ ]:
# Query 9: Ranking categories within each department
# Window function
%%sql
select roots, Category, sum(rating_count) as total_engagement,
rank() over(partition by roots order by sum(rating_count) desc) as Rank_by_engagement
from AMAZON_DATA
group by roots, Category
order by roots, Rank_by_engagement;

 * sqlite:///AmazonDB.db
Done.


roots,Category,total_engagement,Rank_by_engagement
Computers & Accessories,Accessories & Peripherals,5094941.0,1
Computers & Accessories,NetworkingDevices,1401750.0,2
Computers & Accessories,ExternalDevices & DataStorage,1037012.0,3
Computers & Accessories,Components,125025.0,4
Computers & Accessories,"Printers,Inks & Accessories",77579.0,5
Computers & Accessories,Monitors,4637.0,6
Computers & Accessories,Tablets,2886.0,7
Computers & Accessories,Laptops,323.0,8
Electronics,"Headphones,Earbuds & Accessories",4844658.0,1
Electronics,Mobiles & Accessories,4384197.0,2


In [ ]:
#CTEs

%%sql

WITH category_engagement AS (
    SELECT
        roots,
        Category,
        SUM(rating_count) AS total_engagement
    FROM AMAZON_DATA
    GROUP BY roots, Category
)

SELECT
    roots,
    Category,
    total_engagement,
    RANK() OVER (
        PARTITION BY roots
        ORDER BY total_engagement DESC
    ) AS Rank_by_engagement
FROM category_engagement
ORDER BY roots, Rank_by_engagement;

 * sqlite:///AmazonDB.db
Done.


roots,Category,total_engagement,Rank_by_engagement
Computers & Accessories,Accessories & Peripherals,5094941.0,1
Computers & Accessories,NetworkingDevices,1401750.0,2
Computers & Accessories,ExternalDevices & DataStorage,1037012.0,3
Computers & Accessories,Components,125025.0,4
Computers & Accessories,"Printers,Inks & Accessories",77579.0,5
Computers & Accessories,Monitors,4637.0,6
Computers & Accessories,Tablets,2886.0,7
Computers & Accessories,Laptops,323.0,8
Electronics,"Headphones,Earbuds & Accessories",4844658.0,1
Electronics,Mobiles & Accessories,4384197.0,2


In [ ]:
# Highest engagement category and their department using CTs method
%%sql
with category_engagement as (
    select
        roots,
        Category,
        SUM(rating_count) as total_engagement
    from AMAZON_DATA
    group by roots, Category
),

department_rank as (
    select
        roots,
        Category,
        total_engagement,
        RANK() over (
            partition by roots
            order by total_engagement desc
        ) as dept_rank
    from category_engagement
),

top_categories as (
    select *
    from department_rank
    where dept_rank = 1
)

select
    roots,
    Category,
    total_engagement,
    RANK() over (order by total_engagement desc) as overall_rank
from top_categories
order by overall_rank;

 * sqlite:///AmazonDB.db
Done.


roots,Category,total_engagement,overall_rank
Computers & Accessories,Accessories & Peripherals,5094941.0,1
Electronics,"Headphones,Earbuds & Accessories",4844658.0,2
Home & Kitchen,Kitchen & HomeAppliances,2052156.0,3
OfficeProducts,OfficePaperProducts,118700.0,4
Health & PersonalCare,HomeMedicalSupplies & Equipment,3663.0,5


Observation Query 9:

The Computers & Accessories department leads in engagement through Accessories & Peripherals, ranking highest overall among all department-leading categories. Electronics follows closely, while OfficeProducts and Health & PersonalCare show comparatively low engagement dominance.

3. Product Level analysis

Micro-level strategic decisions

Columns used:

product_name

Category

rating

rating_count

discounted_price

In [ ]:
# Query 10: Most reviewed products
%%sql
select Product_type, Category, rating_count, rating, actual_price
from AMAZON_DATA
order by rating_count desc
limit 10;

 * sqlite:///AmazonDB.db
Done.


Product_type,Category,rating_count,rating,actual_price
HDMICables,"HomeTheater,TV & Video",426973.0,4.4,700.0
HDMICables,"HomeTheater,TV & Video",426973.0,4.4,475.0
HDMICables,"HomeTheater,TV & Video",426973.0,4.4,1400.0
HDMICables,"HomeTheater,TV & Video",426972.0,4.4,700.0
HDMICables,"Headphones,Earbuds & Accessories",363713.0,4.1,999.0
HDMICables,"Headphones,Earbuds & Accessories",363713.0,4.1,999.0
HDMICables,"Headphones,Earbuds & Accessories",363711.0,4.1,999.0
WallChargers,Mobiles & Accessories,313836.0,4.1,10999.0
WallChargers,Mobiles & Accessories,313836.0,4.1,8499.0
WallChargers,Mobiles & Accessories,313832.0,4.1,7999.0


In [ ]:
# Query 11: Top 5 products and rating
%%sql
select
    Product_type,
    count(*) as total_products,
    sum(rating_count) as total_engagement,
    avg(rating) as avg_rating
from AMAZON_DATA
group by product_type
order by total_engagement desc
Limit 5;

 * sqlite:///AmazonDB.db
Done.


Product_type,total_products,total_engagement,avg_rating
HDMICables,337,10916828.0,4.057270029673591
USBCables,340,6506933.0,4.159117647058819
WallChargers,111,3383865.0,4.100000000000001
ElectricKettles,290,1902159.0,4.019310344827584
MousePads,58,743844.0,4.186206896551723


In [ ]:
# Query 12: Top 3 Products per category
%%sql
with ranked_products as (
    select
        Product_type,
        Category,
        rating_count,
        rating,
        final_price,
        row_number() over (
            partition by Category
            order by rating_count desc
        ) as rn
    from AMAZON_DATA
)

select *
from ranked_products
where rn <= 3
order by Category, rn;

 * sqlite:///AmazonDB.db
Done.


Product_type,Category,rating_count,rating,final_price,rn
HDMICables,Accessories,205052.0,4.5,939.0,1
HDMICables,Accessories,140036.0,4.3,1149.0,2
HDMICables,Accessories,140036.0,4.3,599.0,3
USBCables,Accessories & Peripherals,178817.0,4.1,709.0,1
USBCables,Accessories & Peripherals,107687.0,4.5,209.0,2
USBCables,Accessories & Peripherals,107686.0,4.5,209.0,3
Tabletop&TravelTripods,Cameras & Photography,93112.0,4.2,2499.0,1
Tabletop&TravelTripods,Cameras & Photography,44696.0,4.3,4499.0,2
BatteryChargers,Cameras & Photography,40895.0,3.8,299.0,3
USBCables,Components,92925.0,4.5,1815.0,1


Creating Tables for further analysis

In [ ]:
%%sql
Drop table if exists department_table;
Create table department_table as
select roots as department, count(*) as total_products, sum(rating_count) as total_engagement,
round(avg(rating),2) as average_rating,
round(sum(rating_count)*100.0/
(select sum(rating_count) from AMAZON_DATA),2) as engagement_share_percentage
from AMAZON_DATA
group by roots;

 * sqlite:///AmazonDB.db
Done.
Done.


[]

In [ ]:
%%sql
select * from department_table

 * sqlite:///AmazonDB.db
Done.


department,total_products,total_engagement,average_rating,engagement_share_percentage
Computers & Accessories,453,7744153.0,4.15,29.04
Electronics,526,15778848.0,4.08,59.17
Health & PersonalCare,1,3663.0,4.0,0.01
Home & Kitchen,448,2991069.0,4.04,11.22
OfficeProducts,31,149675.0,4.31,0.56


In [ ]:
# Department Table
%%sql
Drop table if exists department_table;
Create table department_table as
select roots as department, count(distinct product_id) as total_products, sum(rating_count) as total_engagement,
round(avg(rating),2) as average_rating,
round(avg(final_price),2) as average_final_price,
round(sum(rating_count)*100.0/
(select sum(rating_count) from AMAZON_DATA),2) as engagement_share_percentage
from AMAZON_DATA
group by roots;

 * sqlite:///AmazonDB.db
Done.
Done.


[]

In [ ]:
%%sql
select * from department_table

 * sqlite:///AmazonDB.db
Done.


department,total_products,total_engagement,average_rating,average_final_price,engagement_share_percentage
Computers & Accessories,375,7744153.0,4.15,842.65,29.04
Electronics,490,15778848.0,4.08,5965.89,59.17
Health & PersonalCare,1,3663.0,4.0,899.0,0.01
Home & Kitchen,448,2991069.0,4.04,2330.62,11.22
OfficeProducts,31,149675.0,4.31,301.58,0.56


Ensuring the products do not appear multiple times is crucial, as there is a certainty of counting extra rows when counting all rows. Therefore, DISTINCT is used to accurately measure the number of unique products per department.

In [ ]:
# Category Table
%%sql
DROP TABLE IF EXISTS category_table;

CREATE TABLE category_table AS
SELECT
    Category,

    COUNT(DISTINCT product_id) AS total_products,

    SUM(rating_count) AS total_engagement,

    ROUND(SUM(rating_count) * 1.0
          / COUNT(DISTINCT product_id), 2)
          AS engagement_per_product,

    ROUND(AVG(rating), 2) AS average_rating,

    ROUND(AVG(final_price), 2) AS average_final_price,

    SUM(rating_count * final_price) AS estimated_revenue,

    ROUND(
        SUM(rating_count) * 100.0 /
        (SELECT SUM(rating_count) FROM AMAZON_DATA),
        2
    ) AS engagement_share_percentage,

    ROUND(
        SUM(rating_count * final_price) * 100.0 /
        (SELECT SUM(rating_count * final_price) FROM AMAZON_DATA),
        2
    ) AS revenue_share_percentage

FROM AMAZON_DATA
GROUP BY Category;

 * sqlite:///AmazonDB.db
Done.
Done.


[]

In [ ]:
%%sql
select * from category_table;

 * sqlite:///AmazonDB.db
Done.


Category,total_products,total_engagement,engagement_per_product,average_rating,average_final_price,estimated_revenue,engagement_share_percentage,revenue_share_percentage
Accessories,11,1183177.0,107561.55,4.34,811.86,1010319313.0,4.44,1.41
Accessories & Peripherals,307,5094941.0,16595.9,4.15,486.09,2380882173.4700003,19.11,3.31
Cameras & Photography,16,322657.0,20166.06,4.13,1272.0,557346160.0,1.21,0.78
Components,5,125025.0,25005.0,4.38,1764.4,229164227.0,0.47,0.32
CraftMaterials,7,63214.0,9030.57,4.34,178.57,10785727.0,0.24,0.02
ExternalDevices & DataStorage,18,1037012.0,57611.78,4.32,2151.39,1899599928.0,3.89,2.64
GeneralPurposeBatteries & BatteryChargers,14,190302.0,13593.0,4.35,384.07,106633112.0,0.71,0.15
"Headphones,Earbuds & Accessories",63,4844658.0,76899.33,3.93,948.12,3814073780.0,18.17,5.31
"Heating,Cooling & AirQuality",116,551371.0,4753.2,3.99,3222.32,1582809859.11,2.07,2.2
HomeAudio,16,293704.0,18356.5,4.07,1546.5,369206566.0,1.1,0.51


In [ ]:
%%sql
select
    Category, engagement_share_percentage
FROM category_table
ORDER BY engagement_share_percentage DESC
LIMIT 3;

 * sqlite:///AmazonDB.db
Done.


Category,engagement_share_percentage
Accessories & Peripherals,19.11
"Headphones,Earbuds & Accessories",18.17
Mobiles & Accessories,16.44


Product table

Identify Star Products, Hidden Gems, and Weak Performers.

In [ ]:
%%sql
DROP TABLE IF EXISTS product_table;

CREATE TABLE product_table AS
SELECT
    product_id,
    product_name,
    roots AS department,
    Category,
    Product_type,

    final_price,
    discount_percentage,
    rating,
    rating_count,

    -- Price Segment
    CASE
        WHEN final_price < 1000 THEN 'Budget'
        WHEN final_price BETWEEN 1000 AND 5000 THEN 'Mid-Range'
        ELSE 'Premium'
    END AS price_segment,

    -- Engagement Level
    CASE
        WHEN rating_count > 10000 THEN 'High Engagement'
        WHEN rating_count BETWEEN 1000 AND 10000 THEN 'Medium Engagement'
        ELSE 'Low Engagement'
    END AS engagement_level,

    -- Simple Performance Label
    CASE
        WHEN rating >= 4.2 AND rating_count >= 5000 THEN 'Star Product'
        WHEN rating >= 4.2 AND rating_count < 5000 THEN 'Hidden Gem'
        WHEN rating < 4 AND rating_count >= 5000 THEN 'Mass but Risky'
        ELSE 'Low Performer'
    END AS performance_segment

FROM AMAZON_DATA;

 * sqlite:///AmazonDB.db
Done.
Done.


[]

In [ ]:
%%sql
select * from product_table
limit 5;

 * sqlite:///AmazonDB.db
Done.


product_id,product_name,department,Category,Product_type,final_price,discount_percentage,rating,rating_count,price_segment,engagement_level,performance_segment
B07JW9H4J1,"Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)",Computers & Accessories,Accessories & Peripherals,USBCables,399.0,64,4.2,24269.0,Budget,High Engagement,Star Product
B098NS6PVG,"Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)",Computers & Accessories,Accessories & Peripherals,USBCables,199.0,43,4.0,43994.0,Budget,High Engagement,Low Performer
B096MSW6CT,"Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices",Computers & Accessories,Accessories & Peripherals,USBCables,199.0,90,3.9,7928.0,Budget,Medium Engagement,Mass but Risky
B08HDJ86NZ,"boAt Deuce USB 300 2 in 1 Type-C & Micro USB Stress Resistant, Tangle-Free, Sturdy Cable with 3A Fast Charging & 480mbps Data Transmission, 10000+ Bends Lifespan and Extended 1.5m Length(Martian Red)",Computers & Accessories,Accessories & Peripherals,USBCables,329.0,53,4.2,94363.0,Budget,High Engagement,Star Product
B08CF3B7N1,"Portronics Konnect L 1.2M Fast Charging 3A 8 Pin USB Cable with Charge & Sync Function for iPhone, iPad (Grey)",Computers & Accessories,Accessories & Peripherals,USBCables,154.0,61,4.2,16905.0,Budget,High Engagement,Star Product


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("AmazonDB.db")

pd.read_sql("select * FROM department_table", conn).to_csv("department_table.csv", index=False)
pd.read_sql("select * FROM category_table", conn).to_csv("category_table.csv", index=False)
pd.read_sql("select * FROM product_table", conn).to_csv("product_table.csv", index=False)

conn.close()